In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

pd.set_option("display.float_format", "{:,.4f}".format)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")


## Előfeldolgozás

In [2]:
REF_DATE = pd.to_datetime(train["Policy Start Date"]).max()

cat_cols = train.select_dtypes(include=["object", "string"]).columns.drop("Policy Start Date")
CATEGORIES = {col: sorted(train[col].dropna().unique()) for col in cat_cols}

def preprocess(df):
    df = df.copy()

    date = pd.to_datetime(df["Policy Start Date"])
    df["start_year"] = date.dt.year
    df["start_month"] = date.dt.month
    df["start_dayofweek"] = date.dt.dayofweek
    df["policy_age_days"] = (REF_DATE - date).dt.days
    df = df.drop(columns=["Policy Start Date"])

    for col, cats in CATEGORIES.items():
        df[col] = pd.Categorical(df[col], categories=cats)

    return df

X = preprocess(train.drop(columns=["id", "Premium Amount"]))
y = np.log1p(train["Premium Amount"])
X_test = preprocess(test.drop(columns=["id"]))

print(X.dtypes)

Age                      float64
Gender                  category
Annual Income            float64
Marital Status          category
Number of Dependents     float64
Education Level         category
Occupation              category
Health Score             float64
Location                category
Policy Type             category
Previous Claims          float64
Vehicle Age              float64
Credit Score             float64
Insurance Duration       float64
Customer Feedback       category
Smoking Status          category
Exercise Frequency      category
Property Type           category
start_year                 int32
start_month                int32
start_dayofweek            int32
policy_age_days            int64
dtype: object


## Train/val szétbontás

In [3]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape)

(960000, 22) (240000, 22)


## Baseline

In [4]:
naive_pred = np.full(len(y_val), y_train.mean())
naive_rmsle = np.sqrt(mean_squared_error(y_val, naive_pred))
print(f"Naive baseline RMSLE: {naive_rmsle:.4f}")

Naive baseline RMSLE: 1.0964


## XGBoost modell

In [5]:
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    enable_categorical=True,
    tree_method="hist",
    early_stopping_rounds=100,
    eval_metric="rmse",
    random_state=42,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)

val_pred = model.predict(X_val)
val_rmsle = np.sqrt(mean_squared_error(y_val, val_pred))
print(f"XGBoost validációs RMSLE: {val_rmsle:.4f}")
print(f"Legjobb iteráció: {model.best_iteration}")

[0]	validation_0-rmse:1.09230
[200]	validation_0-rmse:1.04709
[400]	validation_0-rmse:1.04696
[452]	validation_0-rmse:1.04702
XGBoost validációs RMSLE: 1.0469
Legjobb iteráció: 352


## Submission

In [6]:
test_pred = np.expm1(model.predict(X_test))
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({"id": test["id"], "Premium Amount": test_pred})
submission.to_csv("../submissions/02_xgb_baseline.csv", index=False)
submission.head()

,id,Premium Amount
0,1200000,772.0685
1,1200001,767.5752
2,1200002,773.3484
3,1200003,797.4041
4,1200004,727.6379
